# 53 — Executable Monitoring

Convierte los thresholds congelados en M5 en monitoreo ejecutable por batch. Los hallazgos de drift no se ocultan: una alerta `high` bloquea promoción, pero no invalida la capacidad del sistema para detectarla.

In [1]:
from pathlib import Path

from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m5_gate = paths.docs / "decisions" / "M5_GATE.md"
assert m5_gate.exists() and "M5 ✅ CLOSED" in m5_gate.read_text(encoding="utf-8"), (
    "M5 no está formalmente cerrado. No iniciar M6."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
TEMP_DIR = paths.root / ".cache" / "duckdb"
SHADOW_INPUT = paths.root / "data" / "m5" / "fresh_oot_2026.parquet"

assert GOLD_SPEC.schema_version == M6_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M6_SPEC.core.benchmark_model_schema

{
    "m6_schema": M6_SPEC.schema_version,
    "model_version": M6_SPEC.core.model_version,
    "operating_mode": M6_SPEC.core.operating_mode,
    "scoring_mode": M6_SPEC.core.scoring_mode,
    "human_review_required": M6_SPEC.core.human_review_required,
    "production_promotion_authorized": M6_SPEC.deployment.allow_production_promotion,
}


{'m6_schema': 'm6-production-v1',
 'model_version': 'argos-v1',
 'operating_mode': 'shadow',
 'scoring_mode': 'batch',
 'human_review_required': True,
 'production_promotion_authorized': False}

In [2]:
from argos.production.scoring import score_batch
from argos.production.monitoring import BUILD_IMPLEMENTATION, build_monitoring_evidence

scored = score_batch(
    input_path=SHADOW_INPUT, root=paths.root, gold_spec=GOLD_SPEC, modeling_spec=MODEL_SPEC,
    production_spec=M6_SPEC, reports_root=paths.reports, manifest_dir=MANIFEST_DIR, temp_directory=TEMP_DIR,
)
print("implementation:", BUILD_IMPLEMENTATION)
result = build_monitoring_evidence(
    input_path=SHADOW_INPUT,
    scored_path=scored.scored_path,
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.alerts

implementation: m6-53-executable-monitoring-v1


,signal,severity,metric,value,action
0,feature:t0_month,high,psi,8.062556,investigate upstream data and feature distribu...
1,feature:t0_day_of_week,warning,psi,0.106517,investigate upstream data and feature distribu...
2,feature:t0_quarter,high,psi,7.783839,investigate upstream data and feature distribu...
3,feature:ciudad,warning,tvd,0.124450,investigate upstream data and feature distribu...
4,feature:codigo_de_categoria_principal,warning,tvd,0.174850,investigate upstream data and feature distribu...
5,feature:justificacion_modalidad_de_contratacion,warning,tvd,0.116980,investigate upstream data and feature distribu...
6,feature:prior_supplier_contract_count,warning,psi,0.107171,investigate upstream data and feature distribu...
7,feature:prior_entity_contract_count,warning,psi,0.226608,investigate upstream data and feature distribu...
8,feature:supplier_tenure_days,high,psi,0.315983,investigate upstream data and feature distribu...
9,feature:days_since_supplier_last_contract,high,psi,0.340300,investigate upstream data and feature distribu...


In [3]:
result.score_drift

,reference,current_batch,reference_rows,current_rows,reference_mean,current_mean,score_psi,status
0,validation_2024_h2_frozen,a8865bfed9374218183eb302,428076,529054,0.006003,0.011264,1.089164,high


In [4]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,monitoring_feature_coverage,monitoring,True,True,"{""expected"": 41, ""features_monitored"": 41}",every frozen model feature is monitored on eac...
1,monitoring_values_finite,monitoring,True,True,"{""score_psi"": 1.0891642323302304}",feature and score drift metrics are finite
2,alerts_are_actionable,operations,True,True,"{""alerts"": 14, ""high_alerts"": 6}",every warning/high signal is persisted with an...
3,high_alerts_block_promotion,governance,True,True,"{""deployment_state"": {""block_reason"": ""6 high-...",high-severity drift cannot be silently ignored...


In [5]:
blocking = result.validation.loc[result.validation["blocking"]]
print("53 ✅ PASS" if result.ready_for_next_notebook else "53 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("High alerts:", int((result.alerts["severity"] == "high").sum()) if not result.alerts.empty else 0)
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook

53 ✅ PASS
Blocking checks: 4/4
High alerts: 6
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m6_monitoring.json
